# Minute calories and intensity

This notebook does some exploratory data analysis on the following files:
- `minuteCaloriesNarrow_merged.csv`
- `minuteCaloriesWide_merged.csv`
- `minuteIntensitiesNarrow_merged.csv`

This notebook checks missing and duplicate data, looks at intensity by hour, compares the calorie files, and plots calories by hour.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

data_folder = Path('data')
calories_narrow = pd.read_csv(data_folder / 'minuteCaloriesNarrow_merged.csv')
calories_wide = pd.read_csv(data_folder / 'minuteCaloriesWide_merged.csv')
intensities = pd.read_csv(data_folder / 'minuteIntensitiesNarrow_merged.csv')

## Missing values and duplicates

In [ ]:
datasets = {
    'Calories narrow': calories_narrow,
    'Calories wide': calories_wide,
    'Intensity narrow': intensities,
}

for label, frame in datasets.items():
    missing = frame.isna().sum()
    print(f'\n{label}')
    print('Missing values by column:')
    print(missing[missing > 0].to_string() if missing.any() else 'None')
    print(f'Duplicate rows: {frame.duplicated().sum():,}')

**Observation:** The three files have no missing values or duplicate rows, so they appear clean on these basic checks. This does not tell us whether every participant has a complete record for every minute.

In [ ]:
date_format = '%m/%d/%Y %I:%M:%S %p'
intensities['ActivityMinute'] = pd.to_datetime(intensities['ActivityMinute'], format=date_format)

intensities['hour'] = intensities['ActivityMinute'].dt.hour
mean_intensity_by_hour = intensities.groupby('hour')['Intensity'].mean()

fig, ax = plt.subplots()
ax.bar(mean_intensity_by_hour.index, mean_intensity_by_hour)
ax.set(title='Average Intensity by Hour of Day', xlabel='Hour of the day (0-23)', ylabel='Average intensity')
plt.show()

**Observation:** Intensity is generally higher during the day and lower overnight, with its highest hourly average around 18:00.

In [ ]:
minute_columns = [f'Calories{minute:02d}' for minute in range(60)]
calories_from_wide = calories_wide.melt(
    id_vars=['Id', 'ActivityHour'],
    value_vars=minute_columns,
    var_name='minute_column',
    value_name='Calories_wide',
)

hour_start = pd.to_datetime(calories_from_wide['ActivityHour'], format=date_format)
minute_offset = calories_from_wide['minute_column'].str[-2:].astype(int)
calories_from_wide['ActivityMinute'] = hour_start + pd.to_timedelta(minute_offset, unit='min')
calories_from_wide = calories_from_wide[['Id', 'ActivityMinute', 'Calories_wide']]

calories_narrow['ActivityMinute'] = pd.to_datetime(calories_narrow['ActivityMinute'], format=date_format)
calorie_comparison = calories_narrow.merge(
    calories_from_wide,
    on=['Id', 'ActivityMinute'],
    how='outer',
    validate='one_to_one',
)

overlap = calorie_comparison.dropna(subset=['Calories', 'Calories_wide'])
print(f'Overlapping minutes: {len(overlap):,}')
print(f"Overlapping readings match: {overlap['Calories'].eq(overlap['Calories_wide']).all()}")
print(f"Minutes only in narrow file: {calorie_comparison['Calories_wide'].isna().sum():,}")
print(f"Minutes only in wide file: {calorie_comparison['Calories'].isna().sum():,}")

calorie_comparison['Calories'] = calorie_comparison['Calories'].combine_first(
    calorie_comparison['Calories_wide']
)
calorie_comparison['hour'] = calorie_comparison['ActivityMinute'].dt.hour
mean_calories_by_hour = calorie_comparison.groupby('hour')['Calories'].mean()

**Observation:** The two calorie files agree on their overlapping minutes. They also cover some different minutes, so combining the files fills in additional records beyond either file alone.

In [ ]:
fig, ax = plt.subplots()
ax.bar(mean_calories_by_hour.index, mean_calories_by_hour)
ax.set(title='Average Calories per Minute by Hour of Day', xlabel='Hour of the day (0-23)', ylabel='Average calories per minute')
plt.show()

**Observation:** The hourly calorie pattern is similar to the intensity pattern, with higher averages during the day and a peak around 18:00. These are averages across recorded participant-minutes, so they describe this dataset rather than everyone generally.

## Summary statistics

Check the distributions of the calorie and intensity readings.

In [ ]:
print('Calories per minute:')
print(calories_narrow['Calories'].describe().round(2))
print(f"Zero-calorie minutes: {(calories_narrow['Calories'] == 0).sum():,}")

print('\nIntensity:')
print(intensities['Intensity'].describe().round(2))

**Observation:** The mean calorie reading is 1.62 per minute and the median is 1.22. Only 7 of 1,325,580 readings are zero; this check alone does not tell us whether they are valid or recording artefacts, so they are left in the analysis.

In [ ]:
fig, ax = plt.subplots()
ax.hist(calories_narrow['Calories'], bins=20, color='darkorange', edgecolor='black')
ax.set(title='Distribution of Calories per Minute', xlabel='Calories per minute', ylabel='Count')
plt.show()

**Observation:** Most minute-level calorie readings are toward the lower end of the scale, with a smaller number of higher readings.

## Dashboard requirements

### Functional
- Show average calories and intensity by hour.
- Let users filter results by date and participant.
- Compare calories across intensity scores.
- Combine overlapping calorie files without double-counting; show coverage gaps.
- Summarise participants and recorded minutes for the selected filters.

### Non-functional
- Keep charts and filters responsive.
- Ensure calculations are accurate and reproducible.
- Use clear labels, readable contrast and accessible charts.
- Protect participant data and avoid exposing IDs unnecessarily.
- Keep data preparation and dashboard logic easy to maintain.

The functional requirements focus on the comparisons these datasets support: how calories and intensity vary by hour, how they relate, and which records are available for selected dates and participants. Combining the calorie files is useful because their overlapping values match, but each also contains minutes not found in the other.

The non-functional requirements help ensure the dashboard remains responsive and its results are clear, reliable and appropriately protected. Showing recording coverage helps users avoid interpreting unrecorded minutes as zero activity.